In [1]:
from huggingface_hub import whoami
print(whoami())

{'type': 'user', 'id': '69716438e2191b6ccb383827', 'name': 'hshshsuuu', 'fullname': 'lee', 'isPro': False, 'avatarUrl': '/avatars/98d4b03694d0b42358e66feb6049edad.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'independentstudy', 'role': 'fineGrained', 'createdAt': '2026-03-16T13:33:53.173Z', 'fineGrained': {'canReadGatedRepos': True, 'global': [], 'scoped': [{'entity': {'_id': '69716438e2191b6ccb383827', 'type': 'user', 'name': 'hshshsuuu'}, 'permissions': ['repo.content.read', 'repo.access.read', 'repo.write']}]}}}}


In [ ]:
from huggingface_hub import login
token=''
login(token)

In [3]:
from datasets import load_dataset

data=load_dataset('zh-plus/tiny-imagenet')

In [4]:
data

DatasetDict({
    train: Dataset({
        features: ['image', 'label'],
        num_rows: 100000
    })
    valid: Dataset({
        features: ['image', 'label'],
        num_rows: 10000
    })
})

In [ ]:
from src.data import setdata

train=setdata(data['train'])
test=setdata(data['valid'])


In [6]:
from torch.utils.data import DataLoader

train_loader=DataLoader(train,batch_size=32,pin_memory=True,num_workers=4,shuffle=True)
test_loader=DataLoader(test,batch_size=32,pin_memory=True,num_workers=4,shuffle=True)

In [7]:
from transformers import AutoModel

model = AutoModel.from_pretrained(
    "facebook/dinov3-vits16plus-pretrain-lvd1689m"
)

print(model)

Loading weights:   0%|          | 0/235 [00:00<?, ?it/s]

DINOv3ViTModel(
  (embeddings): DINOv3ViTEmbeddings(
    (patch_embeddings): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
  )
  (rope_embeddings): DINOv3ViTRopePositionEmbedding()
  (layer): ModuleList(
    (0-11): 12 x DINOv3ViTLayer(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attention): DINOv3ViTAttention(
        (k_proj): Linear(in_features=384, out_features=384, bias=False)
        (v_proj): Linear(in_features=384, out_features=384, bias=True)
        (q_proj): Linear(in_features=384, out_features=384, bias=True)
        (o_proj): Linear(in_features=384, out_features=384, bias=True)
      )
      (layer_scale1): DINOv3ViTLayerScale()
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): DINOv3ViTGatedMLP(
        (gate_proj): Linear(in_features=384, out_features=1536, bias=True)
        (up_proj): Linear(in_features=384, out_features=1536, bias=True)
        (down_proj): Linear(in_

In [8]:
for p in model.parameters():
    p.requires_grad=True

In [9]:
from src.model import dinosplus_classfier

vit=dinosplus_classfier(model,200)


In [ ]:
import torch.optim as optim
import torch.nn as nn
import torch
optimizer=optim.Adam(vit.parameters(),lr=0.0001)
criterion=nn.CrossEntropyLoss()
scaler=torch.amp.GradScaler()


In [11]:
import torch

torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(False)

In [12]:
print(torch.cuda.is_available())
device='cuda' if torch.cuda.is_available() else 'cpu'


True


In [ ]:
from src.training import train 
train(vit,train_loader,test_loader,50,criterion,scaler,device,optimizer,'full fine tuning vit dino v3 splus','/home/hyuksu/deep-learning-study/outputs/best dino v3 splus.pth')

In [ ]:
from transformers import AutoModel
from src.model import dinosplus_classfier
import torch.optim as optim
import torch
bdino=AutoModel.from_pretrained('facebook/dinov3-vitb16-pretrain-lvd1689m')

for i in bdino.parameters():
    i.required_grad=True
    
model2=dinosplus_classfier(bdino,200)
boptimizer=optim.Adam(model2.parameters(),lr=0.0001)
criterion2=torch.nn.CrossEntropyLoss()




config.json:   0%|          | 0.00/744 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

In [15]:
from src.training import train 
train(model2,train_loader,test_loader,50,criterion2,scaler,device,boptimizer,'full fine tuning vit dino v3 b','/home/hyuksu/deep-learning-study/outputs/best dino v3b.pth')

  0%|          | 0/50 [00:00<?, ?it/s]

Test Loss=5.2984, Test Accuracy=0.0050
Epoch 0 | Train Acc=0.0044 | Train Loss=5.2999
Test Loss=5.2984, Test Accuracy=0.0050
Epoch 1 | Train Acc=0.0046 | Train Loss=5.2994
Test Loss=5.2984, Test Accuracy=0.0050
Epoch 2 | Train Acc=0.0042 | Train Loss=5.2993
Test Loss=5.2984, Test Accuracy=0.0050
Epoch 3 | Train Acc=0.0041 | Train Loss=5.2994
Test Loss=5.2984, Test Accuracy=0.0050
Epoch 4 | Train Acc=0.0044 | Train Loss=5.2994
Test Loss=5.2984, Test Accuracy=0.0050
Epoch 5 | Train Acc=0.0046 | Train Loss=5.2994
Test Loss=5.2984, Test Accuracy=0.0050
Epoch 6 | Train Acc=0.0042 | Train Loss=5.2994
Test Loss=5.2984, Test Accuracy=0.0050
Epoch 7 | Train Acc=0.0045 | Train Loss=5.2994
Test Loss=5.2984, Test Accuracy=0.0050
Epoch 8 | Train Acc=0.0042 | Train Loss=5.2994
Test Loss=5.2984, Test Accuracy=0.0050
Epoch 9 | Train Acc=0.0039 | Train Loss=5.2994
Test Loss=5.2984, Test Accuracy=0.0050
Epoch 10 | Train Acc=0.0045 | Train Loss=5.2993
 Early Stopping Triggered


2026/03/17 04:17:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/17 04:17:57 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


최고 성능 모델이 MLflow에 저장되었습니다.
